### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("gpt-4.1")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots "talk" because they are highly intelligent birds with a strong ability to mimic sounds, including human speech. Here’s a more detailed explanation:\n\n**1. Social nature:**  \nParrots are very social animals in the wild. They live in flocks and constantly communicate with each other using a wide variety of calls and sounds. When kept as pets, parrots treat their human caretakers as part of their flock.\n\n**2. Vocal mimicry:**  \nParrots have a unique voice box called a syrinx, which allows them to reproduce a wide range of sounds. Unlike many birds that sing, parrots are especially skilled at copying noises and voices they hear around them.\n\n**3. Attention and interaction:**  \nParrots often learn to talk because it gets them attention from humans. If you react when your parrot says something, it reinforces the behavior. Talking is a way for the parrot to interact with its environment and companions, both bird and human.\n\n**4. Cognitive stimulation:**  \

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 51, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_abb47f53a9', 'id': 'chatcmpl-EKQhmDKOFjNoBsFwY3lVffWb9EDv1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a06d2a-e3b0-7912-b361-d95586a6f44f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_I2nfykAa5Zq5WOYJbfy4eRb7', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 14, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read

### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)





In [9]:
# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

In [11]:
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's currently sunny in Boston. If you need more details like temperature or a forecast for the week, just let me know!
